# 06. 다변량 분석과 FacetGrid

## 학습 목표
- FacetGrid를 활용한 조건별 비교 분석
- Hue, Size, Style 파라미터로 차원 추가
- Tips 데이터로 팁 패턴의 복합 요인 분석
- 4-5차원 데이터를 2D 평면에 효과적으로 표현

---

## 1. 다변량 시각화의 필요성

**제조업 실무 사례:**
- 라인별·시프트별·제품별 품질 비교 (3개 범주 동시)
- 시간대·요일·지역별 판매량 분석
- 다양한 조건에서의 성능 테스트 결과 비교

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

In [ ]:
# Tips 데이터 로드
tips = sns.load_dataset('tips')
print(f"전체 거래: {len(tips)}건")
print(f"총 팁: ${tips['tip'].sum():.2f}")
print(f"평균 팁 비율: {(tips['tip']/tips['total_bill']).mean()*100:.1f}%")
tips.head()

## 2. Hue: 색상으로 그룹 구분

**비즈니스 질문:** 성별에 따라 계산서-팁 관계가 다른가?

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

# Scatterplot with hue
sns.scatterplot(data=tips, x='total_bill', y='tip', hue='sex', 
                style='sex', s=100, alpha=0.7, ax=ax,
                palette={'Male': '#3A86FF', 'Female': '#FF006E'})

ax.set_xlabel('총 계산서 ($)', fontsize=12)
ax.set_ylabel('팁 ($)', fontsize=12)
ax.set_title('계산서-팁 관계 (성별 비교)', fontsize=14, fontweight='bold')
ax.legend(title='성별', fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# 통계 확인
print("\n성별 팁 비율:")
print(tips.groupby('sex').apply(lambda x: (x['tip']/x['total_bill']).mean() * 100))

# 💡 인사이트: 남성과 여성의 팁 패턴 유사, 차이 미미 (남성 15.8%, 여성 16.6%)

## 3. Size: 점 크기로 추가 변수 표현

**비즈니스 질문:** 테이블 인원수가 팁에 영향을 미치나?

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))

# Scatterplot with hue and size
sns.scatterplot(data=tips, x='total_bill', y='tip', 
                hue='time', size='size', sizes=(50, 400),
                alpha=0.6, ax=ax, palette='Set2',
                edgecolor='black', linewidth=0.5)

ax.set_xlabel('총 계산서 ($)', fontsize=12)
ax.set_ylabel('팁 ($)', fontsize=12)
ax.set_title('계산서-팁 관계 (식사 시간 + 인원수)', fontsize=14, fontweight='bold')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# 💡 인사이트: 큰 원(많은 인원)이 우상단 집중 → 인원 많을수록 계산서·팁 모두 증가
# 💡 Dinner(저녁)에서 고액 거래 많음

## 4. Style: 마커 모양으로 차원 추가

**비즈니스 질문:** 흡연 여부가 팁 패턴에 영향을 주나?

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))

# Scatterplot with hue, size, and style
sns.scatterplot(data=tips, x='total_bill', y='tip',
                hue='day', style='smoker', size='size',
                sizes=(80, 300), alpha=0.7, ax=ax,
                palette='tab10', markers={'Yes': 'X', 'No': 'o'})

ax.set_xlabel('총 계산서 ($)', fontsize=12)
ax.set_ylabel('팁 ($)', fontsize=12)
ax.set_title('계산서-팁 다변량 분석 (요일 + 흡연 + 인원)', fontsize=14, fontweight='bold')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# 💡 인사이트: X 마커(흡연자)와 O 마커(비흡연자) 분포 유사 → 흡연 여부는 팁에 영향 적음
# 💡 주말(Sat, Sun)에 거래 집중

## 5. FacetGrid 기본: 조건별 서브플롯

**비즈니스 질문:** 식사 시간대와 요일 조합별로 패턴이 다른가?

In [ ]:
# FacetGrid 생성: 행=time, 열=sex
g = sns.FacetGrid(tips, row='time', col='sex', height=4, aspect=1.2,
                  margin_titles=True)

g.map(sns.scatterplot, 'total_bill', 'tip', alpha=0.6, s=60, color='#3A86FF',
      edgecolor='black', linewidth=0.5)

g.set_axis_labels('총 계산서 ($)', '팁 ($)', fontsize=11)
g.fig.suptitle('식사 시간 × 성별 팁 패턴', fontsize=14, fontweight='bold', y=1.02)

# 각 서브플롯에 회귀선 추가
g.map(sns.regplot, 'total_bill', 'tip', scatter=False, color='red', line_kws={'linewidth': 2})

plt.tight_layout()
plt.show()

# 💡 인사이트: Dinner(저녁)에서 계산서 범위가 넓고 회귀선 기울기도 가파름
# 💡 Lunch(점심)는 계산서 $10-$25 구간에 집중

## 6. FacetGrid 고급: 다중 그래프 유형

**비즈니스 질문:** 요일별 팁 분포는 어떻게 다른가?

In [ ]:
# FacetGrid with histogram
g = sns.FacetGrid(tips, col='day', col_wrap=2, height=4, aspect=1.3)

g.map(plt.hist, 'tip', bins=15, edgecolor='black', color='#FF006E', alpha=0.7)

g.set_axis_labels('팁 ($)', '빈도', fontsize=11)
g.fig.suptitle('요일별 팁 분포', fontsize=14, fontweight='bold', y=1.01)

# 각 서브플롯에 평균선 추가
for ax, (day_name, day_data) in zip(g.axes.flat, tips.groupby('day')):
    mean_tip = day_data['tip'].mean()
    ax.axvline(mean_tip, color='blue', linestyle='--', linewidth=2.5,
               label=f'평균: ${mean_tip:.2f}')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

# 💡 인사이트: 일요일(Sun) 평균 팁이 가장 높음 ($3.26)
# 💡 토요일(Sat) 거래 건수가 가장 많지만 평균 팁은 중간 수준

## 7. 복합 FacetGrid: Boxplot 활용

**비즈니스 질문:** 흡연 구역과 시간대별로 팁 분포 차이는?

In [ ]:
# FacetGrid with boxplot
g = sns.FacetGrid(tips, col='smoker', row='time', height=4, aspect=1.5,
                  margin_titles=True)

g.map(sns.boxplot, 'day', 'tip', palette='Set2', order=['Thur', 'Fri', 'Sat', 'Sun'])

g.set_axis_labels('요일', '팁 ($)', fontsize=11)
g.fig.suptitle('흡연 여부 × 시간대 × 요일별 팁 분포', fontsize=14, fontweight='bold', y=1.01)

for ax in g.axes.flat:
    ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

# 💡 인사이트: Dinner-Non Smoker-Sun 조합이 가장 높은 팁 중앙값
# 💡 Lunch는 흡연 여부와 무관하게 팁이 낮고 분산도 작음

## 8. 실전 분석: 5차원 데이터 시각화

**종합 분석: 팁 비율 예측 모델 개발을 위한 탐색**

In [ ]:
# 팁 비율 계산
tips['tip_pct'] = tips['tip'] / tips['total_bill'] * 100

fig, ax = plt.subplots(figsize=(14, 8))

# 5차원 표현: x, y, hue, size, style
sns.scatterplot(data=tips, 
                x='total_bill',      # 차원 1
                y='tip_pct',         # 차원 2
                hue='day',           # 차원 3 (색상)
                size='size',         # 차원 4 (크기)
                style='smoker',      # 차원 5 (모양)
                sizes=(100, 500), alpha=0.7, ax=ax,
                palette='tab10', markers={'Yes': 'X', 'No': 'o'},
                edgecolor='black', linewidth=0.8)

ax.set_xlabel('총 계산서 ($)', fontsize=12)
ax.set_ylabel('팁 비율 (%)', fontsize=12)
ax.set_title('5차원 팁 패턴 분석 (계산서 + 팁비율 + 요일 + 인원 + 흡연)', fontsize=14, fontweight='bold')
ax.axhline(20, color='red', linestyle='--', linewidth=2, label='20% 기준선')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# 통계 분석
print("\n팁 비율 요약 통계:")
print(f"평균: {tips['tip_pct'].mean():.2f}%")
print(f"중앙값: {tips['tip_pct'].median():.2f}%")
print(f"표준편차: {tips['tip_pct'].std():.2f}%")
print(f"\n20% 이상 팁: {(tips['tip_pct'] >= 20).sum()}건 ({(tips['tip_pct'] >= 20).mean()*100:.1f}%)")

# 💡 인사이트: 팁 비율은 계산서 금액과 약한 음의 상관 → 소액 거래에서 후한 팁
# 💡 큰 원(많은 인원)에서도 팁 비율 편차 큼 → 인원수만으로 팁 비율 예측 어려움
# 💡 일부 이상치(40% 이상 팁) 존재 → 특별한 서비스 상황?

## 9. 의사결정 지원: 팁 최적화 전략

**비즈니스 질문:** 어떤 조건에서 팁이 가장 좋은가?

In [ ]:
# 조건별 평균 팁 비율 계산
tip_by_conditions = tips.groupby(['time', 'day', 'smoker']).agg({
    'tip_pct': 'mean',
    'tip': 'sum',
    'total_bill': 'count'
}).reset_index()

tip_by_conditions.columns = ['시간', '요일', '흡연', '평균_팁_비율', '총_팁', '거래_건수']
tip_by_conditions = tip_by_conditions.sort_values('평균_팁_비율', ascending=False)

print("\n평균 팁 비율 상위 10개 조건:")
print(tip_by_conditions.head(10))

# 시각화: Top 5 조건
top5 = tip_by_conditions.head(5)
top5['조건'] = top5['시간'] + '-' + top5['요일'] + '-' + top5['흡연'].map({True: '흡연', False: '금연'})

fig, ax = plt.subplots(figsize=(10, 6))

bars = ax.barh(top5['조건'], top5['평균_팁_비율'],
               color=plt.cm.Greens(top5['평균_팁_비율']/top5['평균_팁_비율'].max()),
               edgecolor='black', linewidth=1.5)

# 수치 표시
for i, (cond, pct, cnt) in enumerate(zip(top5['조건'], top5['평균_팁_비율'], top5['거래_건수'])):
    ax.text(pct + 0.3, i, f'{pct:.1f}% ({cnt}건)', 
            va='center', fontsize=10, fontweight='bold')

ax.set_xlabel('평균 팁 비율 (%)', fontsize=12)
ax.set_ylabel('조건 (시간-요일-흡연)', fontsize=12)
ax.set_title('팁 비율 상위 5개 조건', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

# 💡 인사이트: Dinner-Sun-Non Smoker가 최고 팁 비율(19.0%)
# 💡 일요일 저녁 금연 구역에 숙련 서버 배치 권장
# 💡 하지만 거래 건수도 고려해야 → 토요일 저녁의 총 매출 기여도 높음

---

## 📊 핵심 요약

### 다차원 표현 방법 (30%)
| 차원 | 파라미터 | 용도 |
|------|---------|------|
| 1-2 | x, y | 기본 좌표 |
| 3 | hue | 범주형 그룹 (색상) |
| 4 | size | 수치형 크기 |
| 5 | style | 범주형 모양 |
| 6+ | FacetGrid | 서브플롯 분할 |

```python
# 5차원 시각화
sns.scatterplot(x='var1', y='var2', hue='cat1', size='num1', style='cat2')

# FacetGrid
g = sns.FacetGrid(df, row='cat1', col='cat2')
g.map(plot_function, 'x', 'y')
```

### 도출된 인사이트 (70%)
1. **시간 효과**: Dinner가 Lunch보다 계산서 2배, 팁도 비례 증가
2. **요일 패턴**: 일요일 팁 비율 최고(19%), 토요일 거래 건수 최다
3. **인원 효과**: 인원 증가 시 총 팁 증가, 하지만 비율은 감소 경향
4. **흡연 무관**: 흡연 여부는 팁 패턴에 유의미한 영향 없음
5. **역관계**: 계산서가 클수록 팁 비율 감소 → 고액 거래 인센티브 필요

### 실무 의사결정
- **인력 배치**: 일요일 저녁 금연석에 베테랑 서버 배치
- **프로모션**: 소액 거래(< $15) 타겟 이벤트로 빈도 증대
- **테이블 관리**: 대형 테이블(6인 이상)에 팁 가이드라인 제시

### FacetGrid 활용 전략
- 3개 이상 범주형 변수 동시 비교 시 필수
- `col_wrap`으로 자동 줄바꿈 (변수 많을 때)
- `margin_titles=True`로 가독성 향상
- 각 서브플롯에 동일 기준선 추가로 비교 용이

### 주의사항
- 차원이 5개 이상이면 복잡도 급증 → 핵심 변수만 선택
- 색상과 모양 조합 시 색맹 고려 (colorblind-safe palette)
- FacetGrid는 범주형 변수만 가능 → 수치형은 구간화 필요